In [1]:
from itertools import permutations

from cogent3.app.io import open_data_store
from cogent3.app import get_app
from cogent3.evolve.predicate import MotifChange
from cogent3.evolve.ns_substitution_model import NonReversibleDinucleotide

from pathlib import Path

In [2]:
data_dir = Path.cwd().parent / "data"
results_dir = Path.cwd().parent / "results"
fig_dir = Path.cwd().parent / "figures"


In [3]:
def make_gn_preds():
    # making the model parameters (predictates) for
    # the General Nucleotide Markov model
    return [
        MotifChange(f, t, forward_only=True)
        for f, t in permutations("ACTG", 2)
        if f != "T" or t != "G"
    ]

def make_nr_cpg_preds_strand_symetric():
    # same CpG deamination rate on both strands
    # so one parameter
    return [
        # | is the binary or operator that combines the predicates
        # so we have the union of the two changes as a single parameter
        (MotifChange("CG", "TG", forward_only=True) |
        MotifChange("CG", "CA", forward_only=True)) #.aliased("cpg_ts")
    ]

def _make_model(cls, **kwargs):
    return cls(**kwargs)

def GDN_CpG_ss(**kwargs):
    """return a dinucleotide model with strand symmetric CpG deamination"""
    ssym_preds = make_gn_preds() + make_nr_cpg_preds_strand_symetric()
    kwargs=dict(predicates=ssym_preds, optimise_motif_probs=True, name="GDN_CpG_ss")
    return _make_model(NonReversibleDinucleotide, **kwargs)

In [4]:
align_dir = open_data_store("~/repos/mdeq-cpg/data/processed", suffix="fa", mode="r")
out_dstore = open_data_store("~/repos/mdeq-cpg/results/model-results.sqlitedb", mode="a")

In [5]:
tree = "(dyak,(dsim,dmel))"

di_cpg_null = get_app("model", GDN_CpG_ss(), name="null", tree=tree, optimise_motif_probs=True)
di_cpg_alt = get_app("model", GDN_CpG_ss(), name="alt", tree=tree, optimise_motif_probs=True,
                     param_rules = [ {"par_name": "(CG>TG | CG>CA)", "tip_names": ["dyak", "dsim"], "outgroup_name": "dmel", "clade": True},])

In [6]:
loader = get_app("load_aligned", moltype="dna")
hyp = get_app("hypothesis", di_cpg_null, di_cpg_alt)
write_json_app = get_app("write_json", data_store=out_dstore)

In [7]:
app = loader + hyp + write_json_app

In [8]:
import scinexus
scinexus.set_parallel_backend("loky")

In [9]:
app.apply_to(align_dir, parallel=True, show_progress=True,)

DataStoreSqlite(source=/Users/adm_caley/repos/mdeq-cpg/results/model-results.sqlitedb, mode=Mode.a, limit=None, verbose=False)

In [10]:
load_json_app = get_app("load_json")

In [11]:
lf = load_json_app(out_dstore[87])
lf.pvalue

np.float64(0.5928267514898908)